In [0]:
import pandas as pd
import hashlib

dbutils.widgets.text("cleaned_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_cleaned_data.csv")
dbutils.widgets.text("transformed_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_transformed_data.csv")
dbutils.widgets.text("hashed_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_hashed_data.csv")

cleaned_data_file = dbutils.widgets.get("cleaned_data_file")
transformed_data_file = dbutils.widgets.get("transformed_data_file")
hashed_data_file = dbutils.widgets.get("hashed_data_file")

df_cleaned = pd.read_csv(cleaned_data_file, dtype={"block": str})
df_cleaned.shape

In [0]:

df_cleaned["Resale Identifier"] = "S" + df_cleaned['block'].astype(str) \
                                      + df_cleaned['resale_price_avg'].astype(str).str[:2] \
                                      + df_cleaned['month'].astype(str).str[-2:] \
                                      + df_cleaned['town'].astype(str).str[:1]
                                      
df_cleaned.head(10)
                                    

In [0]:
df_cleaned_sorted = df_cleaned.sort_values(by='resale_price', ascending=False)

df_cleaned_dup = df_cleaned_sorted.duplicated(subset=['Resale Identifier'], keep='first')

df_cleaned_unwanted_duplicates = df_cleaned_sorted[df_cleaned_dup].sort_index()
df_transformed = df_cleaned_sorted[~df_cleaned_dup].sort_index()

df_transformed.to_csv(transformed_data_file)
df_transformed.head(10)

In [0]:
df_transformed['hash'] = df_transformed['Resale Identifier'].astype(str).apply(
        lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest()
    )

df_transformed.to_csv(hashed_data_file)

df_transformed.head(10)